## Analyst lesson 4.2 — Value the same borrower from native UFCF


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
from _shared.borrower_model import borrower_model, borrower_vintage
from finstack_quant.core.money import Money
from finstack_quant.statements import ModelBuilder, Evaluator, ForecastSpec, MonteCarloConfig
from finstack_quant.statements_analytics import (
    TerminalValueSpec, evaluate_dcf, dcf_sensitivity, wacc, ScenarioSet, evaluate_scenario_set,
    SensitivityConfig, run_sensitivity, generate_tornado_entries, goal_seek, variance_bridge,
)

borrower = borrower_model()
PERIODS = tuple(period['id'] for period in json.loads(borrower.to_json())['periods'])
borrower_results = Evaluator().evaluate(borrower)
valuation_builder = ModelBuilder.from_spec(borrower)
# Expose operating growth as native formula drivers. Preserve every actual and the baseline forecast.
for node, growth in [("revenue",0.02),("cogs",0.02),("opex",0.015)]:
    valuation_builder.value(f"{node}_growth",[(period,growth) for period in PERIODS])
    valuation_builder = valuation_builder.mixed(node).values(
        [(period,borrower_results.get(node,period)) for period in PERIODS[:4]]
    ).formula(f"lag({node},1) * (1 + {node}_growth)").build()
# The operating graph stores currency-unit scalars. Stamp USD once at the DCF boundary.
valuation_builder.value_money("usd_unit",[(period,Money(1,"USD")) for period in PERIODS])
valuation_builder.compute("ufcf_usd","ufcf * usd_unit")
valuation_builder.compute("ebitda_usd","ebitda * usd_unit")
valuation_builder.compute("total_debt","debt_end * usd_unit").compute("cash","cash_end * usd_unit")
valuation_model = valuation_builder.build()
valuation_results = Evaluator().evaluate(valuation_model)
assert all(abs(valuation_results.get("ufcf",p)-borrower_results.get("ufcf",p)) < 1e-6 for p in PERIODS)
cost_of_capital = wacc(0.65,0.12,0.35,0.075,0.25)
assert abs(cost_of_capital-(0.65*0.12+0.35*0.075*(1-0.25))) < 1e-12
terminal_growth = 0.025
terminal = TerminalValueSpec.gordon_growth(terminal_growth)
base_dcf = evaluate_dcf(valuation_model,cost_of_capital,terminal,ufcf_node="ufcf_usd",as_of="2025-01-01")
assert abs(base_dcf.enterprise_value.amount-base_dcf.net_debt.amount-base_dcf.equity_value.amount) < 0.01
assert abs(base_dcf.net_debt.amount-valuation_results.get("net_debt","2024Q4")) < 0.01
terminal_annual_ufcf = sum(valuation_results.get("ufcf",p) for p in PERIODS[4:])
terminal_undiscounted = terminal_annual_ufcf*(1+terminal_growth)/(cost_of_capital-terminal_growth)
terminal_discount_factor = base_dcf.terminal_value_pv.amount/terminal_undiscounted
explicit_pv = base_dcf.enterprise_value.amount-base_dcf.terminal_value_pv.amount
print("Forecast quarterly UFCF (USD):",[valuation_results.get("ufcf",p) for p in PERIODS[4:]])
print(f"WACC={cost_of_capital:.5%}; annual terminal UFCF={terminal_annual_ufcf:,.2f}")
print(base_dcf.to_json())
print(f"Terminal share of enterprise value={base_dcf.terminal_value_pv.amount/base_dcf.enterprise_value.amount:.2%}")


## Analyst lesson 4.2 — Native DCF sensitivity and an operating tornado


In [ ]:
from finstack_quant.reporting.charts import tornado_chart
from finstack_quant.reporting.theme import INSTITUTIONAL
from IPython.display import SVG, display

rows = []
for discount_rate in (0.0876875,0.0976875,0.1076875):
    for growth in (0.015,0.025,0.035):
        value = evaluate_dcf(valuation_model,discount_rate,TerminalValueSpec.gordon_growth(growth),ufcf_node="ufcf_usd")
        rows.append({"WACC":discount_rate,"terminal_growth":growth,"equity_usd":value.equity_value.amount})
grid = pd.DataFrame(rows).pivot(index="WACC",columns="terminal_growth",values="equity_usd")
assert (grid.diff(axis=0).iloc[1:] < 0).all().all()
assert (grid.diff(axis=1).iloc[:,1:] > 0).all().all()
print(grid.to_string())
dcf_rank = dcf_sensitivity(valuation_model,cost_of_capital,terminal,ufcf_node="ufcf_usd",wacc_sensitivity_bump=0.01)
assert not dcf_rank.wacc_down_clamped and not dcf_rank.terminal_growth_up_clamped
print("Native EV deltas under lower/higher parameter shocks:\n",dcf_rank.to_dataframe().to_string(index=False))
# The chart groups impacts by economic sign; the table above preserves parameter direction.
dcf_svg = tornado_chart([(entry.parameter_id,min(entry.downside,entry.upside)/1e6,max(entry.downside,entry.upside)/1e6)
                         for entry in dcf_rank.entries],theme=INSTITUTIONAL)
Path("dcf-tornado-usd-millions.svg").write_text(dcf_svg)
display(SVG(dcf_svg))
operating_sensitivity = run_sensitivity(valuation_model,SensitivityConfig(mode="tornado",
    parameters=[("revenue_growth","2025Q4",0.02,[-0.02,0.06]),("opex_growth","2025Q4",0.015,[-0.015,0.045])],target_metrics=["ufcf"]))
operating_entries = generate_tornado_entries(operating_sensitivity,"ufcf","2025Q4")
assert len(operating_entries)==2
print("Operating sensitivity is quarterly UFCF, not EV:",[(e.parameter_id,e.downside,e.upside) for e in operating_entries])
operating_svg = tornado_chart([(e.parameter_id,min(e.downside,e.upside)/1e3,max(e.downside,e.upside)/1e3) for e in operating_entries],theme=INSTITUTIONAL)
Path("operating-tornado-usd-thousands.svg").write_text(operating_svg)
display(SVG(operating_svg))


## Analyst lesson 4.2 — Price bull/base/bear native statement paths


In [ ]:
scenario_set = ScenarioSet({"base":{},"bull":{"revenue_growth":0.04},"bear":{"revenue_growth":-0.02}})
scenario_results = evaluate_scenario_set(valuation_model,scenario_set)

def price_ufcf_path(forecast_values):
    """Carry one native quarterly UFCF path into the native DCF without mixing marginal percentiles."""
    money_values = [(p,Money(valuation_results.get("ufcf",p) if p in PERIODS[:4] else float(forecast_values[p]),"USD")) for p in PERIODS]
    priced_model = ModelBuilder.from_spec(valuation_model).value_money("ufcf_usd",money_values).build()
    return evaluate_dcf(priced_model,cost_of_capital,terminal,ufcf_node="ufcf_usd")

scenario_rows = []
for name in scenario_results.names:
    result = scenario_results.get(name)
    assert all(result.get("revenue",p)==valuation_results.get("revenue",p) for p in PERIODS[:4])
    value = price_ufcf_path({p:result.get("ufcf",p) for p in PERIODS[4:]})
    scenario_rows.append({"scenario":name,"2025Q4_UFCF_usd":result.get("ufcf","2025Q4"),"equity_usd":value.equity_value.amount})
scenario_table = pd.DataFrame(scenario_rows).set_index("scenario")
assert scenario_table.loc["bull","equity_usd"] > scenario_table.loc["base","equity_usd"] > scenario_table.loc["bear","equity_usd"]
assert abs(scenario_table.loc["base","equity_usd"]-base_dcf.equity_value.amount) < 0.01
print(scenario_table.to_string())


## Analyst lesson 4.2 — Solve a price-implied terminal assumption


In [ ]:
goal_builder = ModelBuilder.from_spec(valuation_model)
goal_builder.value("wacc_assumption",[(p,cost_of_capital) for p in PERIODS])
goal_builder.value("terminal_growth_assumption",[(p,terminal_growth) for p in PERIODS])
goal_builder.compute("ufcf_ttm","ufcf + lag(ufcf,1) + lag(ufcf,2) + lag(ufcf,3)")
goal_builder.compute("terminal_ev_hook","ufcf_ttm * usd_unit * (1+terminal_growth_assumption)/(wacc_assumption-terminal_growth_assumption)")
goal_model = goal_builder.build()
market_equity = 130_000_000.0
market_enterprise = market_equity + base_dcf.net_debt.amount
required_terminal = (market_enterprise-explicit_pv)/terminal_discount_factor
implied = goal_seek(goal_model,"terminal_ev_hook","2025Q4",required_terminal,
                    "terminal_growth_assumption","2025Q4",True,(-0.02,0.045))
repriced = evaluate_dcf(valuation_model,cost_of_capital,TerminalValueSpec.gordon_growth(implied.solved_value),ufcf_node="ufcf_usd")
assert abs(repriced.equity_value.amount-market_equity) < 0.01
print(f"Target equity={market_equity:,.2f}; implied perpetual growth={implied.solved_value:.6%}; native reprice={repriced.equity_value.amount:,.2f}")


## Analyst lesson 4.2 — Reconcile Gordon growth with an exit multiple


In [ ]:
exit_ebitda = sum(valuation_results.get("ebitda", p) for p in PERIODS[4:])
equivalent_multiple = terminal_undiscounted / exit_ebitda
exit_value = evaluate_dcf(valuation_model,cost_of_capital,TerminalValueSpec.exit_multiple(equivalent_multiple),
                         ufcf_node="ufcf_usd",exit_multiple_metric_node="ebitda_usd")
assert abs(exit_value.enterprise_value.amount-base_dcf.enterprise_value.amount) < 0.01
print(f"Equivalent exit EV / trailing-year EBITDA={equivalent_multiple:.6f}x")
print(f"Gordon EV={base_dcf.enterprise_value.amount:,.2f}; exit-multiple EV={exit_value.enterprise_value.amount:,.2f}")
print("The multiple is implied by this cash-conversion ratio, discount rate and growth; it is not an independent market comparable.")


## Analyst lesson 4.2 — Explain revised UFCF with additive signed components


In [ ]:
vintage_results = []
for vintage in ("2024Q3","2024Q4"):
    builder = ModelBuilder.from_spec(borrower_vintage(vintage))
    builder.compute("after_tax_ebit","ebit * (1-tax_rate)")
    builder.compute("capex_outflow","-capex").compute("nwc_outflow","-change_nwc")
    vintage_results.append(Evaluator().evaluate(builder.build()))
vintage_bridge = variance_bridge(vintage_results[0],vintage_results[1],"ufcf","2025Q4",
    ["after_tax_ebit","depreciation","capex_outflow","nwc_outflow"],"Q3 information","Q4 information")
assert abs(vintage_bridge.unexplained) < 1e-6
print(vintage_bridge.to_json())
print("Raw driver deltas reconcile because these nodes are additive signed UFCF components. Revenue alone is not a causal EV attribution.")


## Analyst lesson 4.2 — Value retained joint statement paths


In [ ]:
import matplotlib.pyplot as plt

stochastic_model = ModelBuilder.from_spec(valuation_model).forecast(
    "revenue",ForecastSpec.log_normal(math.log(1.02),0.025,17)).build()
mc_config = MonteCarloConfig(1024,17,[0.05,0.5,0.95],True)
statement_mc = Evaluator().evaluate_monte_carlo(stochastic_model,mc_config)
retained = statement_mc.to_paths_dataframe()
ufcf_paths = retained.loc[retained.metric=="ufcf_usd"].pivot(index="path_id",columns="period",values="value")
assert ufcf_paths.shape==(1024,4) and not ufcf_paths.isna().any().any()
enterprise_values = np.asarray([price_ufcf_path(row).enterprise_value.amount for _,row in ufcf_paths.iterrows()])
assert np.isfinite(enterprise_values).all() and enterprise_values.std(ddof=1)>0
print("Native statement paths / native DCF EV p5,p50,p95 (USD):",np.quantile(enterprise_values,[0.05,0.5,0.95]))
print("Distribution of modeled outcomes, not a confidence interval for a known fair value. WACC, terminal growth and costs are fixed.")
fig, ax = plt.subplots(figsize=(8,3.6))
ax.hist(enterprise_values/1e6,bins=35,color="#167d77",edgecolor="white")
ax.axvline(base_dcf.enterprise_value.amount/1e6,color="#a45227",label="Deterministic base")
ax.set(xlabel="Enterprise value (USD millions)",ylabel="Simulated paths",title="Revenue-growth uncertainty through the borrower UFCF graph")
ax.legend();fig.tight_layout()
